In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
data = fetch_california_housing()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [ ]:
def build_model(optimizer):
    model = models.Sequential([
        layers.Dense(32, activation='relu', input_shape=(X_train.shape[1],)),
        layers.Dense(16, activation='relu'),
        layers.Dense(1)
    ])
    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
    return model

In [ ]:
optimizers_list = {
    "SGD": optimizers.SGD(learning_rate=0.01),
    "Momentum": optimizers.SGD(learning_rate=0.01, momentum=0.9),
    "RMSProp": optimizers.RMSprop(learning_rate=0.01),
    "Adam": optimizers.Adam(learning_rate=0.01)
}

histories = {}

for name, opt in optimizers_list.items():
    print(f"🔹 Training with {name} optimizer...")
    model = build_model(opt)
    hist = model.fit(X_train, y_train, epochs=60, batch_size=64,
                     validation_split=0.2, verbose=0)
    histories[name] = hist.history

🔹 Training with SGD optimizer...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



🔹 Training with Momentum optimizer...
🔹 Training with RMSProp optimizer...
🔹 Training with Adam optimizer...


In [ ]:
fig = go.Figure()

for name, hist in histories.items():
    epochs = np.arange(len(hist['loss']))
    fig.add_trace(go.Scatter3d(
        x=epochs,
        y=hist['loss'],
        z=hist['val_loss'],
        mode='lines',
        name=name,
        line=dict(width=6)
    ))

fig.update_layout(
    title='📉 Comparison of Optimizers in Training (3D View)',
    scene=dict(
        xaxis_title='Epoch',
        yaxis_title='Training Loss',
        zaxis_title='Validation Loss'
    ),
    width=950, height=700
)

fig.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))
for name, hist in histories.items():
    plt.plot(hist['val_loss'], label=f'{name}')
plt.title('Optimizer Comparison (Validation Loss)')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:

import numpy as np
import plotly.graph_objects as go

def loss_fn(w1, w2):

    return w1**2 + 0.5*w2**2 + 0.1*w1*w2

def grad_fn(w1, w2):
    dw1 = 2*w1 + 0.1*w2
    dw2 = w2 + 0.1*w1
    return dw1, dw2

def optimize(method, lr=0.1, epochs=50):
    w1, w2 = np.random.randn(), np.random.randn()
    path = [(w1, w2, loss_fn(w1, w2))]

    v1, v2 = 0, 0
    s1, s2 = 0, 0
    beta1, beta2 = 0.9, 0.999
    eps = 1e-8

    for t in range(1, epochs+1):
        dw1, dw2 = grad_fn(w1, w2)

        if method == "SGD":
            w1 -= lr * dw1
            w2 -= lr * dw2

        elif method == "Momentum":
            v1 = beta1 * v1 + (1 - beta1) * dw1
            v2 = beta1 * v2 + (1 - beta1) * dw2
            w1 -= lr * v1
            w2 -= lr * v2

        elif method == "Adam":
            v1 = beta1 * v1 + (1 - beta1) * dw1
            v2 = beta1 * v2 + (1 - beta1) * dw2
            s1 = beta2 * s1 + (1 - beta2) * (dw1**2)
            s2 = beta2 * s2 + (1 - beta2) * (dw2**2)
            v1_corr = v1 / (1 - beta1**t)
            v2_corr = v2 / (1 - beta1**t)
            s1_corr = s1 / (1 - beta2**t)
            s2_corr = s2 / (1 - beta2**t)
            w1 -= lr * v1_corr / (np.sqrt(s1_corr) + eps)
            w2 -= lr * v2_corr / (np.sqrt(s2_corr) + eps)

        path.append((w1, w2, loss_fn(w1, w2)))
    return np.array(path)

w1_vals = np.linspace(-3, 3, 100)
w2_vals = np.linspace(-3, 3, 100)
W1, W2 = np.meshgrid(w1_vals, w2_vals)
Z = loss_fn(W1, W2)

paths = {
    "SGD": optimize("SGD"),
    "Momentum": optimize("Momentum"),
    "Adam": optimize("Adam")
}

fig = go.Figure()

fig.add_trace(go.Surface(
    x=W1, y=W2, z=Z,
    colorscale='Viridis', opacity=0.7, showscale=False,
    name="Loss Surface"
))

colors = {"SGD": "red", "Momentum": "orange", "Adam": "cyan"}

for name, path in paths.items():
    fig.add_trace(go.Scatter3d(
        x=path[:,0],
        y=path[:,1],
        z=path[:,2],
        mode='lines+markers',
        line=dict(width=6, color=colors[name]),
        marker=dict(size=4),
        name=name
    ))

fig.update_layout(
    title='🧭 مسیر حرکت بهینه‌سازها روی سطح خطا (Loss Surface)',
    scene=dict(
        xaxis_title='Weight 1 (w1)',
        yaxis_title='Weight 2 (w2)',
        zaxis_title='Loss'
    ),
    width=950, height=750
)

fig.show()